In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

In [ ]:
def quadrupole_kepler_derivs(t, state, epsilon):
    """
    Equations of motion for a Kepler problem perturbed by an external 
    quadrupole potential: V_quad = -0.5 * epsilon * (2*z^2 - x^2 - y^2)
    """
    x, y, z, vx, vy, vz = state
    r2 = x**2 + y**2 + z**2
    r3 = r2**1.5
    
    # Pure Kepler acceleration + quadrupole perturbation forces
    ax = -x / r3 + epsilon * x
    ay = -y / r3 + epsilon * y
    az = -z / r3 - 2.0 * epsilon * z
    
    return [vx, vy, vz, ax, ay, az]

In [ ]:
def generate_poincare_section(epsilon=0.015, n_orbits=15):
    """Integrates several orbits with different initial conditions to map the phase space."""
    
    # Define the equatorial crossing event (z = 0)
    def z_crossing_event(t, state, epsilon):
        return state[2]  # Returns the z coordinate
    
    # We only want crossings going upward (v_z > 0)
    z_crossing_event.direction = 1
    z_crossing_event.terminal = False
    
    plt.figure(figsize=(10, 8))
    np.random.seed(101)  # For reproducible visual distribution
    
    # Loop over various initial conditions to sample islands and chaotic regions
    for i in range(n_orbits):
        # We launch from x0, y=0, z=0 on the equatorial plane
        x0 = np.random.uniform(0.7, 1.3)
        # Give it a mix of azimuthal and vertical velocity
        vy0 = np.random.uniform(0.5, 0.8)
        vz0 = np.random.uniform(0.1, 0.4)
        
        # Calculate vx0 to keep total energy bounded/similar if desired, 
        # here we keep it simple with vx0 = 0
        vx0 = 0.0
        
        init_state = [x0, 0.0, 0.0, vx0, vy0, vz0]
        t_span = (0, 3000)  # Long integration time to collect many punch points
        
        # Integrate the orbit
        sol = solve_ivp(
            quadrupole_kepler_derivs, 
            t_span, 
            init_state, 
            args=(epsilon,), 
            events=z_crossing_event,
            max_step=0.1,  # Keep steps small for symplectic-like accuracy
            rtol=1e-8, 
            atol=1e-8
        )
        
        # Extract the state variables at the moments of equatorial crossing
        if sol.y_events and len(sol.y_events[0]) > 0:
            crossings = sol.y_events[0]
            
            # Compute spherical coordinates or cylindrical invariants on the section
            # We will plot cylindrical radius R vs R_dot (radial velocity v_R)
            x_c = crossings[:, 0]
            y_c = crossings[:, 1]
            vx_c = crossings[:, 3]
            vy_c = crossings[:, 4]
            
            R = np.sqrt(x_c**2 + y_c**2)
            v_R = (x_c * vx_c + y_c * vy_c) / R
            
            # Plot this unique orbit's punch marks with its own distinct color
            plt.scatter(R, v_R, s=2.5, alpha=0.8, label=f"Orbit {i+1}" if i < 5 else "")

    # plt.title(f"Poincaré Surface of Section ($z=0, v_z>0$)\n3D Kepler Problem with External Quadrupole Perturbation ($\epsilon = {epsilon}$)", fontsize=13)
    plt.xlabel(r"Cylindrical Radius $R = \sqrt{x^2 + y^2}$")
    plt.ylabel(r"Radial Velocity $v_R$")
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()

In [ ]:
# Run the visualization
generate_poincare_section(epsilon=0.015, n_orbits=18)